# Whose Africa Is in the Public Domain?

**Computational Text Analysis Pack · Module 01 — Framing the Question**

*Technique:* Corpus construction as an argument; the absence audit for text

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/00-foundations/01b_whose_africa_is_public_domain.ipynb)

Data: `colonial_texts.csv`, `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv', 'masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

Every corpus is a claim about what counts as evidence. This one examines the
claim built into the corpus you have been handed — and it is uncomfortable.

`colonial_texts.csv` was built by searching Project Gutenberg for "africa" and
taking what came back. Gutenberg holds work that is out of copyright, which for
practical purposes means published before 1929.

Ask what that filter selects for, before running anything.

## The tooling

No library does this notebook's work: deciding what a corpus can answer is
judgement, not computation. **pandas** is used only to look at what we have.

In [ ]:
print(f"{len(df_colonial):,} passages from {df_colonial['work_id'].nunique()} works")
print()
print("Most represented authors:")
print(df_colonial['author'].value_counts().head(8).to_string())
print()
print("This is a COLONIAL-DISCOURSE corpus, not an African-authored one:")
print("it comes from a Project Gutenberg keyword search for 'africa', so it is")
print("mostly European and American writing ABOUT Africa before 1929.")

## Read the author list again

These are, overwhelmingly, European and American writers. Equiano and Douglass
are there — formerly enslaved authors writing against the grain — but most of
the collection is colonial administrators, missionaries and travel writers
describing a continent to readers at home.

The corpus is not "African literature before 1929". It is **writing about Africa,
in English, by whoever was being published in Europe and America.**

In [ ]:
# The filter is copyright, and copyright has a date.
# African literature in English is overwhelmingly a post-1950 phenomenon —
# after independence, and therefore still in copyright.
timeline = [
    (1789, "Equiano, The Interesting Narrative", "in this corpus"),
    (1899, "Conrad, Heart of Darkness", "in this corpus"),
    (1952, "Tutuola, The Palm-Wine Drinkard", "IN COPYRIGHT"),
    (1958, "Achebe, Things Fall Apart", "IN COPYRIGHT"),
    (1986, "Soyinka wins the Nobel Prize", "IN COPYRIGHT"),
    (2013, "Adichie, Americanah", "IN COPYRIGHT"),
]
print(f"{'year':>6}  {'work':<44}{'status'}")
print("-" * 74)
for y, w, s in timeline:
    print(f"{y:>6}  {w:<44}{s}")
print()
print("The public domain contains the colonial archive and almost none of the")
print("literature written in response to it. A corpus built from what is freely")
print("available inherits that shape.")

### 🔧 Your turn 1

Count how many distinct authors are in `df_colonial`, and look at the ten most
represented.

How many would you describe as African? Write the number down — you will use it
in your limits section.

## What this means in practice

It does not mean the corpus is useless. It means the questions it can answer are
about **colonial discourse**, and any finding must be phrased that way.

- Defensible: *"In this corpus of pre-1929 English-language writing about Africa,
  the word 'native' appears N times per thousand, most often alongside …"*
- Not defensible: *"African literature uses the word 'native' …"*

The first is a finding about a documented archive. The second attributes European
writing to African authors.

In [ ]:
import re

def count_words(texts):
    counts = Counter()
    for t in texts:
        counts.update(re.findall(r"\w+", str(t).lower()))
    return counts

# A vocabulary that shows what the corpus is.
counts_col = count_words(df_colonial["text"])
total_col = sum(counts_col.values())
for term in ("native", "savage", "civilisation", "civilization", "tribe", "african"):
    n = counts_col[term]
    print(f"  {term:<16}{n:>6} occurrences   {n / total_col * 1000:>6.2f} per 1,000")
print()
print("This vocabulary is the corpus telling you what it is.")

In [ ]:
# The contrast that makes the point: the same words in contemporary
# African journalism, written by Africans.
counts_news = count_words(df[df["language"] == "English"]["text"])
total_news = sum(counts_news.values())

print(f"{'word':<16}{'colonial /1k':>14}{'African news /1k':>20}")
print("-" * 50)
for term in ("native", "tribe", "savage", "civilisation", "development", "government"):
    a = counts_col[term] / total_col * 1000
    b = counts_news[term] / total_news * 1000
    print(f"{term:<16}{a:>14.2f}{b:>20.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
terms = ["native", "tribe", "savage", "civilisation", "development", "government"]
col = [counts_col[t] / total_col * 1000 for t in terms]
news = [counts_news[t] / total_news * 1000 for t in terms]
x = np.arange(len(terms))
ax.bar(x - 0.2, col, 0.4, label="pre-1929 writing about Africa")
ax.bar(x + 0.2, news, 0.4, label="contemporary African news (English)")
ax.set_xticks(x); ax.set_xticklabels(terms, rotation=20)
ax.set_ylabel("occurrences per 1,000 words")
ax.set_title("Two corpora, two vocabularies for the same continent")
ax.legend()
plt.tight_layout(); plt.show()

### 🔧 Your turn 2

Add two words of your own to that comparison — one you expect to be colonial,
one you expect to be contemporary.

Were you right? A prediction you got wrong is worth more than one you got right:
it is the corpus telling you something you did not already believe.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Fewer than a handful. Equiano and Douglass are the clear cases,
and both wrote from enslavement rather than from Africa. The overwhelming
majority of the 150 works are by European and American authors. Write that
number into your limits section verbatim — it constrains every claim the corpus
can support.

**Your turn 2.** The predictable result is that *native*, *tribe* and *savage*
are far denser in the colonial corpus, while *development* and *government*
dominate contemporary news. The interesting part is usually a word you expected
to be colonial that turns out common in both, or vice versa — that is where a
continuity worth writing about hides.

Note carefully what this comparison is **not**: contemporary Nigerian journalism
and Victorian travel writing differ in period, genre, purpose, audience and
authorship all at once. You cannot attribute the difference to any one of those.
It is a contrast that raises a question, not a test that answers one.

</details>

## Where this stops

You know what your corpus is and what it silently excludes. That is the absence
audit, done for text — and it is worth as much of your final mark here as it was
in the data-science course.

Next: **Module 02 — describing a corpus**.